#  Data Type Conversion and Type Casting in PySpark

## Convert String, Integer, Decimal, Date and Timestamp Columns


## 1. What Is Data Type Conversion?

Data type conversion means changing a column from one type to another.

```text
"5000"                  → integer
"45000.75"              → double or decimal
"2026-07-29"            → date
"2026-07-29 10:30:00"   → timestamp
```

The value may look correct, but Spark must know its actual type before calculations or date operations.

## 2. Why Is Type Casting Required?

Data from CSV, JSON, APIs and databases may arrive in the wrong format.

For example, a salary column may be stored as text:

```text
"45000"
"52000"
```

Before calculating total or average salary, convert it to a numeric type.

## 3. Windows Local Setup

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

## 4. Create SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Type_Casting_in_PySpark") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 6. Check Existing Data Types

Use `printSchema()` for a readable schema.

In [3]:
data = [
    (1, "Amit",  "28",      "45000.50", "2026-01-15", "2026-01-15 09:30:00"),
    (2, "Priya", "31",      "52000.75", "2026-02-20", "2026-02-20 10:15:00"),
    (3, "Rahul", "invalid", "48000.00", "20-03-2026", "2026-03-20 11:00:00"),
    (4, "Sneha", "27",      "NA",       "2026-04-10", "invalid"),
    (5, "Karan", None,      "60000.25", None,         None)
]

columns = [
    "employee_id", "employee_name", "age",
    "salary", "joining_date", "login_time"
]

df = spark.createDataFrame(data, columns)
df.show()

+-----------+-------------+-------+--------+------------+-------------------+
|employee_id|employee_name|    age|  salary|joining_date|         login_time|
+-----------+-------------+-------+--------+------------+-------------------+
|          1|         Amit|     28|45000.50|  2026-01-15|2026-01-15 09:30:00|
|          2|        Priya|     31|52000.75|  2026-02-20|2026-02-20 10:15:00|
|          3|        Rahul|invalid|48000.00|  20-03-2026|2026-03-20 11:00:00|
|          4|        Sneha|     27|      NA|  2026-04-10|            invalid|
|          5|        Karan|   NULL|60000.25|        NULL|               NULL|
+-----------+-------------+-------+--------+------------+-------------------+



In [4]:
df.printSchema()

root
 |-- employee_id: long (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- joining_date: string (nullable = true)
 |-- login_time: string (nullable = true)



Use `dtypes` to get a Python list of column names and types.

In [6]:
df.dtypes

[('employee_id', 'bigint'),
 ('employee_name', 'string'),
 ('age', 'string'),
 ('salary', 'string'),
 ('joining_date', 'string'),
 ('login_time', 'string')]

## 7. Basic `cast()` Syntax

```python
col("column_name").cast("new_data_type")
```

Normally, it is used inside `withColumn()`:

```python
df = df.withColumn(
    "age",
    col("age").cast("integer")
)
```

In [5]:
from pyspark.sql.functions import col

## 8. Convert String to Integer

Use valid values first to understand the basic syntax.

In [8]:
age_df = spark.createDataFrame(
    [(1, "25"), (2, "30"), (3, "35")],
    ["id", "age"]
)

age_df = age_df.withColumn(
    "age",
    col("age").cast("integer")
)

age_df.show()
age_df.printSchema()

+---+---+
| id|age|
+---+---+
|  1| 25|
|  2| 30|
|  3| 35|
+---+---+

root
 |-- id: long (nullable = true)
 |-- age: integer (nullable = true)



## 9. Convert String to Double

In [9]:
salary_df = spark.createDataFrame(
    [(1, "45000.50"), (2, "52000.75")],
    ["id", "salary"]
)

salary_df = salary_df.withColumn(
    "salary",
    col("salary").cast("double")
)

salary_df.show()
salary_df.printSchema()

+---+--------+
| id|  salary|
+---+--------+
|  1| 45000.5|
|  2|52000.75|
+---+--------+

root
 |-- id: long (nullable = true)
 |-- salary: double (nullable = true)



## 10. Convert String to Decimal

For exact decimal precision, especially monetary values, use:

```text
decimal(10,2)
```

- `10` = maximum total digits
- `2` = digits after the decimal point

In [10]:
decimal_df = spark.createDataFrame(
    [(1, "45000.50"), (2, "52000.75")],
    ["id", "salary"]
)

decimal_df = decimal_df.withColumn(
    "salary",
    col("salary").cast("decimal(10,2)")
)

decimal_df.show()
decimal_df.printSchema()

+---+--------+
| id|  salary|
+---+--------+
|  1|45000.50|
|  2|52000.75|
+---+--------+

root
 |-- id: long (nullable = true)
 |-- salary: decimal(10,2) (nullable = true)



## 11. Double vs Decimal

| Type | Best use |
|---|---|
| `double` | Approximate decimal calculations |
| `decimal(p,s)` | Exact decimal precision, such as financial data |

## 12. Convert Numeric Column to String

Identifiers may contain digits but should not be treated as quantities.

In [11]:

id_string_df = df.withColumn(
    "employee_id",
    col("employee_id").cast("string")
)

id_string_df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- joining_date: string (nullable = true)
 |-- login_time: string (nullable = true)



## 13. Invalid Values During Casting

The original data contains values such as:

```text
age = "invalid"
salary = "NA"
```

These cannot be converted directly to numbers.

Depending on Spark version and ANSI settings, an invalid cast may return null or raise an error. A safer approach is to validate the value before casting.

## 14. Safely Convert Age to Integer

Use `rlike()` to check whether the value contains only digits.

In [13]:
from pyspark.sql.functions import when

df_safe_age = df.withColumn(
    "age_integer",
    when(
        col("age").rlike(r"^[0-9]+$"),
        col("age").cast("integer")
    ).otherwise(None)
)

df_safe_age.select(
    "employee_name", "age", "age_integer"
).show()

+-------------+-------+-----------+
|employee_name|    age|age_integer|
+-------------+-------+-----------+
|         Amit|     28|         28|
|        Priya|     31|         31|
|        Rahul|invalid|       NULL|
|        Sneha|     27|         27|
|        Karan|   NULL|       NULL|
+-------------+-------+-----------+



### Understanding the Pattern

```python
r"^[0-9]+$"
```

## Regex Explanation: `r"^[0-9]+$"`

This regular expression checks whether a value contains **only digits from beginning to end**.

| Part | Meaning |  Explanation|
|---|---|---|
| `^` | Start anchor | Matching must start from the beginning of the value |
| `[0-9]` | Digit range | Allows any digit from `0` to `9` |
| `+` | One or more | At least one digit must be present |
| `$` | End anchor | Matching must continue until the end of the value |

### Combined Meaning

```text
The complete value must contain one or more digits only.
means the complete value must contain one or more digits.

```text
"28"      → valid
"31"      → valid
"invalid" → invalid
```


## 15. Safely Convert Salary to Double

In [14]:
df_safe_salary = df.withColumn(
    "salary_double",
    when(
        col("salary").rlike(r"^[0-9]+(\.[0-9]+)?$"),
        col("salary").cast("double")
    ).otherwise(None)
)

df_safe_salary.select(
    "employee_name", "salary", "salary_double"
).show()

+-------------+--------+-------------+
|employee_name|  salary|salary_double|
+-------------+--------+-------------+
|         Amit|45000.50|      45000.5|
|        Priya|52000.75|     52000.75|
|        Rahul|48000.00|      48000.0|
|        Sneha|      NA|         NULL|
|        Karan|60000.25|     60000.25|
+-------------+--------+-------------+



| Part     | Meaning               | Simple explanation                                    |
| -------- | --------------------- | ----------------------------------------------------- |
| `^`      | Start of value        | Matching must begin from the first character          |
| `[0-9]`  | Any digit             | Allows digits from `0` to `9`                         |
| `+`      | One or more           | At least one digit is required                        |
| `(` `)`  | Group                 | Groups the decimal portion together                   |
| `\.`     | Literal decimal point | Matches an actual `.` character                       |
| `[0-9]+` | Digits after decimal  | Requires one or more digits after `.`                 |
| `?`      | Optional              | The complete decimal part may appear zero or one time |
| `$`      | End of value          | Matching must continue to the final character         |


## 16. Convert String to Date

Use:

```python
to_date(column, format)
```

In [7]:
from pyspark.sql.functions import to_date, try_to_date

date_df = spark.createDataFrame(
    [(1, "2026-01-15"), (2, "2026-02-20")],
    ["id", "joining_date"]
)

date_df = date_df.withColumn(
    "joining_date",
    to_date(col("joining_date"), "yyyy-MM-dd")
)

date_df.show()
date_df.printSchema()

+---+------------+
| id|joining_date|
+---+------------+
|  1|  2026-01-15|
|  2|  2026-02-20|
+---+------------+

root
 |-- id: long (nullable = true)
 |-- joining_date: date (nullable = true)



## 17. Common Date and Time Symbols

| Symbol | Meaning |
|---|---|
| `yyyy` | Four-digit year |
| `MM` | Month |
| `dd` | Day |
| `HH` | Hour in 24-hour format |
| `mm` | Minute |
| `ss` | Second |

Examples:

```text
2026-07-29 → yyyy-MM-dd
29-07-2026 → dd-MM-yyyy
29/07/2026 → dd/MM/yyyy
```

## 18. Handle More Than One Date Format

The original data contains both:

```text
2026-01-15
20-03-2026
```

Use `coalesce()` to return the first successful conversion.

In [10]:
from pyspark.sql.functions import coalesce, when

df_dates = df.withColumn(
    "joining_date_converted",
    coalesce(
        try_to_date(col("joining_date"), "yyyy-MM-dd"),
        try_to_date(col("joining_date"), "dd-MM-yyyy")
    )
)

df_dates.select(
    "employee_name", "joining_date", "joining_date_converted"
).show()

+-------------+------------+----------------------+
|employee_name|joining_date|joining_date_converted|
+-------------+------------+----------------------+
|         Amit|  2026-01-15|            2026-01-15|
|        Priya|  2026-02-20|            2026-02-20|
|        Rahul|  20-03-2026|            2026-03-20|
|        Sneha|  2026-04-10|            2026-04-10|
|        Karan|        NULL|                  NULL|
+-------------+------------+----------------------+



## 21. Convert Multiple Columns Together

Chain multiple `withColumn()` operations.

In [11]:

df_converted = df \
    .withColumn(
        "age",
        when(
            col("age").rlike(r"^[0-9]+$"),
            col("age").cast("integer")
        ).otherwise(None)
    ) \
    .withColumn(
        "salary",
        when(
            col("salary").rlike(r"^[0-9]+(\.[0-9]+)?$"),
            col("salary").cast("decimal(10,2)")
        ).otherwise(None)
    ) \
    .withColumn(
        "joining_date",
        coalesce(
            try_to_date(col("joining_date"), "yyyy-MM-dd"),
            try_to_date(col("joining_date"), "dd-MM-yyyy")
        )
    )

df_converted.show()

+-----------+-------------+----+--------+------------+-------------------+
|employee_id|employee_name| age|  salary|joining_date|         login_time|
+-----------+-------------+----+--------+------------+-------------------+
|          1|         Amit|  28|45000.50|  2026-01-15|2026-01-15 09:30:00|
|          2|        Priya|  31|52000.75|  2026-02-20|2026-02-20 10:15:00|
|          3|        Rahul|NULL|48000.00|  2026-03-20|2026-03-20 11:00:00|
|          4|        Sneha|  27|    NULL|  2026-04-10|            invalid|
|          5|        Karan|NULL|60000.25|        NULL|               NULL|
+-----------+-------------+----+--------+------------+-------------------+



## 22. Verify the Final Schema

Always check the schema after conversion.

In [12]:
df_converted.printSchema()

root
 |-- employee_id: long (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: decimal(10,2) (nullable = true)
 |-- joining_date: date (nullable = true)
 |-- login_time: string (nullable = true)



Expected types:

```text
age          → integer
salary       → decimal
joining_date → date
login_time   → timestamp
```

In [13]:
spark.stop()